In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv
/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv


In [2]:
movies_dataset = os.path.join(dirname, "tmdb_5000_movies.csv")
credits_dataset = os.path.join(dirname, "tmdb_5000_credits.csv")

movies = pd.read_csv(movies_dataset)
credits = pd.read_csv(credits_dataset)


In [3]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [4]:
credits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4803 non-null   int64 
 1   title     4803 non-null   object
 2   cast      4803 non-null   object
 3   crew      4803 non-null   object
dtypes: int64(1), object(3)
memory usage: 150.2+ KB


In [5]:
movies = movies.merge(credits, on='title')
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [6]:
# genres 
# id 
# keywords
# title
# overview
# cast
# crew
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

In [7]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4809 non-null   int64 
 1   title     4809 non-null   object
 2   overview  4806 non-null   object
 3   genres    4809 non-null   object
 4   keywords  4809 non-null   object
 5   cast      4809 non-null   object
 6   crew      4809 non-null   object
dtypes: int64(1), object(6)
memory usage: 263.1+ KB


In [8]:
movies.dropna(inplace=True)

In [9]:
movies.duplicated().sum()

np.int64(0)

In [10]:
import ast

def fetch_name(obj):
    return [item['name'] for item in ast.literal_eval(obj)]

def fetch_director(obj):
    return [item['name'] for item in ast.literal_eval(obj) if item['job'] == 'Director']

In [11]:
movies['genres'] = movies['genres'].apply(fetch_name)
movies['keywords'] = movies['keywords'].apply(fetch_name)
movies['cast'] = movies['cast'].apply(fetch_name)
movies['crew'] = movies['crew'].apply(fetch_director)

In [12]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [13]:
movies['genres'] = movies['genres'].apply(lambda x:[item.replace(' ', '') for item in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[item.replace(' ', '') for item in x])
movies['cast'] = movies['cast'].apply(lambda x:[item.replace(' ', '') for item in x])
movies['crew'] = movies['crew'].apply(lambda x:[item.replace(' ', '') for item in x])

In [14]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [15]:
new_df = movies[['movie_id', 'title', 'tags']]

In [16]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

/tmp/ipykernel_16/3089450492.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))


In [17]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

/tmp/ipykernel_16/3214958533.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())


In [18]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [19]:
new_df['tags'][0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver stephenlang michellerodriguez giovanniribisi joeldavidmoore cchpounder wesstudi lazalonso dileeprao mattgerald seananthonymoran jasonwhyte scottlawrence kellykilgour jamespatrickpitt seanpatrickmurphy peterdillon kevindorman kelsonhenderson davidvanhorn jacobtomuri michaelblain-rozgay joncurry lukehawker woodyschultz petermensah soniayee jahnelcurfman ilramchoi kylawarren lisaroumain debrawilson chrismala taylorkibby jodielandau julielamm cullenb.madden josephbradymadden frankietorres austinwilson sarawilson tamicawashington-miller lucybriant nathanm

In [20]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')

In [21]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [22]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0])

In [23]:
print(len(cv.get_feature_names_out()))
features = cv.get_feature_names_out()

pd.Series(features)

5000


0                  000
1                   10
2                  100
3                   11
4                   12
             ...      
4995           zombies
4996              zone
4997               zoo
4998    zooeydeschanel
4999        zoëkravitz
Length: 5000, dtype: object

In [24]:
import nltk

from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [25]:
def stem(words):
    y = []

    for word in words.split():
        y.append(ps.stem(word))

    return " ".join(y)

In [26]:
new_df['tags'] = new_df['tags'].apply(stem)

/tmp/ipykernel_16/3213734980.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [27]:
new_df['tags'][0]

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav stephenlang michellerodriguez giovanniribisi joeldavidmoor cchpounder wesstudi lazalonso dileeprao mattgerald seananthonymoran jasonwhyt scottlawr kellykilgour jamespatrickpitt seanpatrickmurphi peterdillon kevindorman kelsonhenderson davidvanhorn jacobtomuri michaelblain-rozgay joncurri lukehawk woodyschultz petermensah soniaye jahnelcurfman ilramchoi kylawarren lisaroumain debrawilson chrismala taylorkibbi jodielandau julielamm cullenb.madden josephbradymadden frankietorr austinwilson sarawilson tamicawashington-mil lucybri nathanmeist gerryblair matthewchamberlain paulyat wraywil

In [28]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [29]:
# list(enumerate(similarity[0]))
pd.Series(sorted(list(enumerate(similarity[0]))))

0                           (0, 1.0)
1           (1, 0.07142857142857144)
2           (2, 0.05143444998736397)
3           (3, 0.03327791628198609)
4           (4, 0.16101529717988267)
                    ...             
4801    (4801, 0.023002185311411807)
4802    (4802, 0.048795003647426664)
4803    (4803, 0.023262105259961773)
4804    (4804, 0.025717224993681984)
4805                     (4805, 0.0)
Length: 4806, dtype: object

In [30]:
def recommend_movies(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    print(f"movie_index: {movie_index}")
    
    distances = similarity[movie_index]
    print(f"distances: {distances}")

    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x:x[1])[1:10]
    print(f" ")

    for movie in movies_list:
        print(f"{new_df.iloc[movie[0]].title}: {movie[0]}")

In [31]:
recommend_movies("Avatar")

movie_index: 0
distances: [1.         0.07142857 0.05143445 ... 0.02326211 0.02571722 0.        ]
 
Lifeforce: 1916
Aliens vs Predator: Requiem: 1214
Battle: Los Angeles: 582
Titan A.E.: 539
Independence Day: 507
Krull: 1440
Apollo 18: 3606
Independence Daysaster: 4190
Ender's Game: 260


In [32]:
import pickle

with open("movies_recommended_system.pkl", "wb") as file:
    pickle.dump(new_df, file)